In [ ]:
import glob
import os
import numpy as np
import pandas as pd
import scienceplots
import seaborn as sns
import yaml
from matplotlib import pyplot as plt
from tqdm import tqdm
from scipy import stats
from matplotlib.ticker import PercentFormatter
import marsilea as ma
import marsilea.plotter as mp
from pybedtools import BedTool
plt.style.use(["science", "nature"])
import pickle

plt.rcParams['xtick.labelsize'] = 5
plt.rcParams['ytick.labelsize'] = 5
plt.rcParams['axes.labelsize'] = 6
plt.rcParams["xtick.top"] = False
plt.rcParams["ytick.right"] = False
plt.rcParams["lines.linewidth"] = 0.5
plt.rcParams["legend.fontsize"] = 6
plt.rcParams['hatch.linewidth'] = 0.5

# %matplotlib widget

protocol_map = {
    "Visium": "10X Visium",
    "VisiumHD": "10X Visium HD",
    "Chromium": "10X Chromium",
    "Dropseq": "Drop-seq",
    "Stereoseq": "Stereo-seq",
    "Slideseq": "Slide-seq V2",
    "SpatialTranscriptomics": "ST",
    "Microwell": "Microwell-seq",
    "annotation": "Annotated PAS",
    "Annotation": "Annotated PAS",
    "anno": "Annotated PAS",
}
type_map = {
    "Visium": "Spatial transcriptome",
    "VisiumHD": "Spatial transcriptome",
    "Chromium": "scRNA-seq",
    "Dropseq": "scRNA-seq",
    "Stereoseq": "Spatial transcriptome",
    "Slideseq": "Spatial transcriptome",
    "SpatialTranscriptomics": "Spatial transcriptome",
    "Microwell": "scRNA-seq",
}
order = ["10X Chromium", "Drop-seq", "Microwell-seq", "10X Visium","Stereo-seq", "Slide-seq V2", "ST"]
# order = ["10X Chromium", "Drop-seq", "Microwell-seq", "10X Visium", "10X Visium HD","Stereo-seq", "Slide-seq V2", "Spatial Transcriptomics"]

color = [
    "#386b98",
    "#269a51",
    "#edaa4d",
    "#d34123",
    "#7e648a",
    "#454545",
    "#929292",
]
palette=sns.color_palette(color, 7)
mm = 1/25.4

In [ ]:
import json
import glob
import pandas as pd
from collections import defaultdict

def process_json_files(model_result_path):
    """处理JSON结果文件并生成结构化DataFrame"""
    
    # 初始化数据容器
    data_dict = defaultdict(list)
    sample_ids = []
    
    # 遍历所有JSON文件
    for idx, json_file in enumerate(glob.glob(model_result_path + "*.json")):
        with open(json_file) as f:
            data = json.load(f)
            
        # 生成样本ID（可根据需要自定义）
        sample_id = "_".join(os.path.basename(json_file).split('.')[0].split("_")[0:4])
        sample_ids.append(sample_id)
        
        data_size = int(data["data_size"])
        apex_mean = float(data["apex_mean"])
        apex_std = float(data["apex_std"])
        weighted_mean = float(data["weighted_mean"])
        weighted_std = float(data["weighted_std"])


        # 提取自定义模型数据
        for prefix in ["custom_", "normal_"]:
            model_type = "custom_model" if prefix == "custom_" else "normal_dist"
            
            # 参数提取
            params = data[model_type]["params"]
            for param, value in params.items():
                data_dict[f"{prefix}{param}"].append(value)
                
            # 指标提取
            for metric in ["log_likelihood", "AIC", "BIC"]:
                data_dict[f"{prefix}{metric}"].append(data[model_type][metric])
        
        data_dict["data_size"].append(data_size)
        data_dict["apex_mean"].append(apex_mean)
        data_dict["apex_std"].append(apex_std)
        data_dict["weighted_mean"].append(weighted_mean)
        data_dict["weighted_std"].append(weighted_std)

    # 转换为DataFrame
    df = pd.DataFrame(data_dict)
    df.insert(0, "sample_id", sample_ids)  # 添加样本ID列
    
    # 列排序（可选）
    column_order = ["sample_id"] 
    column_order += [f"custom_{x}" for x in ["mu1","sigma1","sigma2","a","b","log_likelihood","AIC","BIC"]]
    column_order += [f"normal_{x}" for x in ["mu","sigma","log_likelihood","AIC","BIC"]]
    column_order += ["data_size","apex_mean","apex_std","weighted_mean","weighted_std"]
    
    return df[column_order]

# 使用示例
model_result_path = "/path/to/apabenchmark_final/data/model_result/"
combined_df = process_json_files(model_result_path)



In [ ]:
combined_df["protocol"] = combined_df["sample_id"].str.split("_").str[0]
combined_df["tissue"] = combined_df["sample_id"].str.split("_").str[2]
combined_df["species"] = combined_df["sample_id"].str.split("_").str[1]
combined_df["delta_AIC_custom"] = combined_df["custom_AIC"] - combined_df[["custom_AIC", "normal_AIC"]].min(axis=1)
combined_df["delta_AIC_normal"] = combined_df["normal_AIC"] - combined_df[["custom_AIC", "normal_AIC"]].min(axis=1)

In [ ]:
protocol_counts = combined_df.groupby('protocol')['species'].nunique()
valid_protocols = protocol_counts[protocol_counts == 2].index.tolist()
analysis_df = combined_df[combined_df['protocol'].isin(valid_protocols)].copy()

In [ ]:
analysis_df = analysis_df.reset_index(drop=True)

In [ ]:
combined_df["delta_AIC"] = combined_df["custom_AIC"] - combined_df["normal_AIC"]
combined_df["delta_BIC"] = combined_df["custom_BIC"] - combined_df["normal_BIC"]

In [ ]:
# use x.xx e xx
print(f"Mean delta AIC: {format(combined_df['delta_AIC'].mean(), '.2e')}")
print(f"Mean delta BIC: {format(combined_df['delta_BIC'].mean(), '.2e')}")

In [ ]:
from tqdm import tqdm
import statsmodels.formula.api as smf
from statsmodels.tools.sm_exceptions import ConvergenceWarning
import warnings
warnings.simplefilter('ignore', ConvergenceWarning)

def bootstrap_lrt(null_formula, full_formula, group_var, re_formula, data, B=100):
    # 拟合原假设模型
    null_model = smf.mixedlm(null_formula, data=data, groups=group_var, re_formula=re_formula)
    null_result = null_model.fit(reml=False, maxiter=2000)  # 增加收敛设置
    
    # 拟合全模型
    full_model = smf.mixedlm(full_formula, data=data, groups=group_var, re_formula=re_formula)
    full_result = full_model.fit(reml=False, maxiter=2000)
    
    # 计算观察到的LRT统计量
    lrt_observed = -2 * (null_result.llf - full_result.llf)
    
    # 准备设计矩阵
    X_null = null_model.exog 
    fixef = null_result.fe_params
    
    # 获取方差分量
    ranef_var = null_result.cov_re.values[0][0] if null_result.cov_re is not None else 0
    resid_var = null_result.scale
    
    # 确保随机效应方差非负
    ranef_var = max(ranef_var, 0)
    
    # 改进的Bootstrap流程
    lrt_bootstrap = []
    max_retries = 100  # 最大重试次数
    
    for _ in tqdm(range(B), desc="Bootstrapping"):
        success = False
        retries = 0
        
        while not success and retries < max_retries:
            try:
                # 生成随机效应
                groups = data[group_var]
                unique_groups = groups.unique()
                group_effects = {grp: np.random.normal(0, np.sqrt(ranef_var)) for grp in unique_groups}
                re = groups.map(group_effects).values
                
                # 生成响应变量
                y = X_null.dot(fixef) + re + np.random.normal(0, np.sqrt(resid_var), len(data))
                
                # 创建bootstrap数据集
                boot_df = data.copy()
                boot_df["value"] = y
                
                # 重新拟合模型（不使用start_params）
                null_boot = smf.mixedlm(null_formula, data=boot_df, groups=group_var, re_formula=re_formula)
                null_boot_result = null_boot.fit(reml=False, maxiter=10000, method='powell')  # 动态截取参数
                
                full_boot = smf.mixedlm(full_formula, data=boot_df, groups=group_var, re_formula=re_formula)
                full_boot_result = full_boot.fit(reml=False, maxiter=10000, method='powell')
                
                # 计算LRT
                lrt = -2 * (null_boot_result.llf - full_boot_result.llf)
                # 严格收敛检查
                if (null_boot_result.converged 
                    and full_boot_result.converged
                    and lrt >= 0
                    ):                   
                    lrt = max(-2 * (null_boot_result.llf - full_boot_result.llf), 0)
                    lrt_bootstrap.append(lrt)
                    success = True
                else:
                    retries += 1
                    
            except Exception as e:
                print(f"Error: {e}")
                retries += 1
                continue
        if not success:
            print(f"Failed after {max_retries} retries")
    
    # 计算p值（添加连续性修正）
    if len(lrt_bootstrap) == 0:
        return lrt_observed, np.nan
    pval = (np.sum(np.array(lrt_bootstrap) >= lrt_observed) + 1) / (len(lrt_bootstrap) + 1)
    return lrt_observed, pval


# species_lrt, species_p = bootstrap_lrt(
#     null_formula="value ~ protocol",
#     full_formula="value ~ species * protocol",
#     group_var="tissue",
#     re_formula="1",
#     data=analysis_df,
#     B=1000
# )
# print(f"Species LRT: {species_lrt}, p-value: {species_p}")

# protocol_lrt, protocol_p = bootstrap_lrt(
#     null_formula="value ~ species",
#     full_formula="value ~ species * protocol",
#     group_var="tissue",
#     re_formula="1",
#     data=analysis_df,
#     B=1000
# )

# print(f"Protocol LRT: {protocol_lrt}, p-value: {protocol_p}")

# interaction_lrt, interaction_p = bootstrap_lrt(
#     null_formula="value ~ species + protocol",
#     full_formula="value ~ species * protocol",
#     group_var="tissue",
#     re_formula="1",
#     data=analysis_df,
#     B=1000
# )

# print(f"Interaction LRT: {interaction_lrt}, p-value: {interaction_p}")




def bootstrap_lrt_random(data, group_var="tissue", B=1000):
    # 预处理数据类型
    data = data.copy()
    data[group_var] = data[group_var].astype(str)
    
    # 拟合原假设模型 (OLS)
    null_formula = "value ~ species * protocol"
    null_model = smf.ols(null_formula, data=data)
    null_result = null_model.fit()
    
    # 拟合备择模型 (混合效应)
    full_formula = "value ~ species * protocol"
    full_model = smf.mixedlm(full_formula, 
                           groups=data[group_var],
                           re_formula="1",
                           data=data)
    full_result = full_model.fit(reml=False, method='lbfgs', maxiter=3000)
    
    # 计算观察到的LRT统计量
    lrt_observed = max(-2 * (null_result.llf - full_result.llf), 0)
    
    # 准备参数
    X = null_model.exog
    fixef = null_result.params
    resid_var = max(null_result.mse_resid, 1e-6)  # 方差下限
    
    # 改进的Bootstrap流程
    lrt_bootstrap = []
    max_retries = 5  # 最大重试次数
    
    for _ in tqdm(range(B), desc="Testing Random Effect"):
        success = False
        retries = 0
        
        while not success and retries < max_retries:
            try:
                # 生成新数据 (基于原假设)
                y = X.dot(fixef) + np.random.normal(0, np.sqrt(resid_var), len(X))
                
                # 创建数据集
                boot_df = data.copy()
                boot_df["value"] = y
                
                # 拟合原假设模型
                null_boot = smf.ols(null_formula, data=boot_df)
                null_boot_result = null_boot.fit()
                
                # 拟合备择模型（带参数初始化）
                full_boot = smf.mixedlm(full_formula, 
                                      groups=boot_df[group_var],
                                      re_formula="1",
                                      data=boot_df)
                
                full_boot_result = full_boot.fit(
                    reml=False,
                    method='nm',
                    maxiter=3000,
                )
                
                # 严格收敛检查
                lrt = max(-2 * (null_boot_result.llf - full_boot_result.llf), 0)
                if (full_boot_result.converged and lrt >= 0):
                    lrt_bootstrap.append(lrt)
                    success = True
                else:
                    retries += 1
                    
            except Exception as e:
                retries += 1
                print(f"Retry {retries}: {e}")
                continue
                
        if not success:
            print(f"Failed after {max_retries} retries")

    # 计算p值
    if len(lrt_bootstrap) == 0:
        return lrt_observed, np.nan
    
    pval = (np.sum(np.array(lrt_bootstrap) >= lrt_observed) + 1) / (len(lrt_bootstrap) + 1)
    return lrt_observed, pval

# # 使用示例
# tissue_lrt, tissue_p = bootstrap_lrt_random(analysis_df, B=1000)
# print(f"Tissue Random Effect: LRT={tissue_lrt:.2f}, p={tissue_p:.4f}")

In [ ]:
import pandas as pd

# 定义需要检验的指标列表
metrics = [
    'custom_mu1', 'custom_sigma1', 'custom_sigma2',
    'custom_a', 'custom_b', 'weighted_mean', 'weighted_std'
]

# 创建结果存储结构
results = []

# 遍历每个指标进行检验
for metric in metrics:
    print(f"\n{'='*40}")
    print(f"Processing metric: {metric}")
    print(f"{'='*40}")
    
    # 创建当前指标的副本数据
    current_df = analysis_df.copy()
    current_df["value"] = current_df[metric]
    
    try:
        # 物种效应检验
        species_lrt, species_p = bootstrap_lrt(
            null_formula="value ~ protocol",
            full_formula="value ~ species * protocol",
            group_var="tissue",
            re_formula="1",
            data=current_df,
            B=1000
        )
        
        # 方案效应检验
        protocol_lrt, protocol_p = bootstrap_lrt(
            null_formula="value ~ species",
            full_formula="value ~ species * protocol",
            group_var="tissue",
            re_formula="1",
            data=current_df,
            B=1000
        )
        
        # 交互效应检验
        interaction_lrt, interaction_p = bootstrap_lrt(
            null_formula="value ~ species + protocol",
            full_formula="value ~ species * protocol",
            group_var="tissue",
            re_formula="1",
            data=current_df,
            B=1000
        )
        
        # 组织随机效应检验
        tissue_lrt, tissue_p = bootstrap_lrt_random(
            data=current_df, 
            group_var="tissue",
            B=1000
        )
        
        # 记录结果
        results.append({
            'Metric': metric,
            'Species_LRT': species_lrt,
            'Species_p': species_p,
            'Protocol_LRT': protocol_lrt,
            'Protocol_p': protocol_p,
            'Interaction_LRT': interaction_lrt,
            'Interaction_p': interaction_p,
            'Tissue_LRT': tissue_lrt,
            'Tissue_p': tissue_p
        })
        
        # 打印当前结果
        print(f"\n{metric} Results:")
        print(f"Species: LRT={species_lrt:.2f}, p={species_p:.4f}")
        print(f"Protocol: LRT={protocol_lrt:.2f}, p={protocol_p:.4f}")
        print(f"Interaction: LRT={interaction_lrt:.2f}, p={interaction_p:.4f}")
        print(f"Tissue RE: LRT={tissue_lrt:.2f}, p={tissue_p:.4f}")
        
    except Exception as e:
        print(f"Error processing {metric}: {str(e)}")
        results.append({
            'Metric': metric,
            'Error': str(e)
        })

# 转换为DataFrame并保存结果
results_df = pd.DataFrame(results)

# 添加异常处理列
if 'Error' not in results_df:
    results_df['Error'] = None

# 保存结果到CSV
results_df.to_csv("bootstrap_lrt_results.csv", index=False)

# 打印整理后的结果
print("\n\nFinal Results:")
print(results_df[['Metric', 'Species_p', 'Protocol_p', 'Interaction_p', 'Tissue_p']].to_markdown(index=False))

# 返回结果DataFrame
results_df


In [ ]:
results_df.to_csv("../data/peak_paramsbootstrap_lrt_results.csv", index=False)